# Lecture 8 Lab — Retrieval-Augmented Generation (RAG)

## Every step of RAG, made visible on a corpus of Ren Zhengfei's speeches

Lecture 8 is about **RAG (Retrieval-Augmented Generation)**: instead of changing the model's weights, you **retrieve** a small amount of relevant evidence at question time and let the model answer *based on that evidence*. This lab takes the pipeline apart so you can run every stage yourself:

```text
corpus  →  chunk  →  index  →  retrieve  →  audit the evidence  →  grounded answer with citations
```

In this lab you will:
1. **Audit a knowledge base** — 405 public speeches/interviews by Ren Zhengfei (任正非, founder of Huawei), 1994–2019.
2. Match questions to the most relevant speech chunks with a **zero-dependency TF-IDF retriever**.
3. Produce an offline answer with **`[year · title]` citations**, auditable sentence by sentence.
4. **Implement a minimal retriever yourself**, to see what TF-IDF is really doing.
5. *(Optional)* compare **semantic embedding retrieval** and **naive LLM vs RAG**.

**A note on language:** the corpus is Chinese, so the ten discussion questions and the retrieval queries are Chinese too — translating them would break keyword retrieval. Every question comes with an English gloss, and all retrieved evidence carries `[year · title]` citations, so you can follow the full pipeline without reading Chinese.

> **Prerequisite:** finish `Lec01_02_Lab_Getting_Started.ipynb` first (it sets up Python via the Tsinghua mirror). The **default path of this lab uses only the Python standard library** — no GPU, no API key, no network, no extra packages. If an *optional* section needs a package, install via the Tsinghua mirror, e.g.:
> `pip install -i https://pypi.tuna.tsinghua.edu.cn/simple sentence-transformers`

*One-sentence intuition:* RAG doesn't make the model "smarter" — it **changes the evidence the model can see**, and **constrains** the answer to an auditable corpus.

In [1]:
# Quick environment check — standard library only
import sys
from pathlib import Path
from collections import Counter

# make rag_ren.py / questions.py in this folder importable
ROOT = Path.cwd()
if not (ROOT / "rag_ren.py").exists():
    raise RuntimeError("Start this notebook from the Lec08_RAG_Lab folder (it contains rag_ren.py).")
sys.path.insert(0, str(ROOT))

import rag_ren
from questions import QUESTIONS

CORPUS_DIR = rag_ren.DEFAULT_CORPUS_DIR
BUILD_DIR = ROOT / "build"
BUILD_DIR.mkdir(exist_ok=True)

print("Python       :", sys.version.split()[0])
print("corpus dir   :", CORPUS_DIR)
print("corpus found :", CORPUS_DIR.is_dir())
if not CORPUS_DIR.is_dir():
    print("\n⚠️  Corpus not found. Three fixes (any one works):")
    print("   1) use the full course repo (corpus lives in MiniCourse_8hr/demos/Ren-master/);")
    print("   2) copy Ren-master into this folder and rename it corpus/;")
    print("   3) set the env var REN_CORPUS_DIR to the corpus directory.")
else:
    print("✅ environment ready — let's start the Lecture 8 lab")

Python       : 3.9.18
corpus dir   : /u/zfeng2/Github/AI-ML-2026/AI-ECON-2026/labs/Lec08_RAG_Lab/corpus
corpus found : True
✅ environment ready — let's start the Lecture 8 lab


## Step 1 — Audit the corpus: "what knowledge did we give the system?"

The first step of RAG is not calling a model — it is understanding what the knowledge base actually contains. Run the cell below to see the corpus size and year span.

In [2]:
docs = rag_ren.load_corpus(CORPUS_DIR)          # [(relative path, year, title, body), ...]
year_counts = Counter(year for _rel, year, _title, _text in docs)
char_count = sum(len(text) for _rel, _year, _title, text in docs)

print(f"documents : {len(docs)}")
print(f"characters: {char_count:,}")
print(f"years     : {min(year_counts)}-{max(year_counts)} ({len(year_counts)} distinct years)")

# one sample: title + first 120 characters (Chinese - it's a Chinese corpus)
rel, year, title, text = docs[len(docs)//2]
print(f"\nsample speech -> [{year} · {title}]")
print("  ", text[:120].replace(chr(10), " "), "...")

documents : 405
characters: 1,578,927
years     : 1994-2019 (26 distinct years)

sample speech -> [2014 · 在华大建设思路汇报会上的讲话]
   ## 在华大建设思路汇报会上的讲话  2014年3月27日，5月12日    【导  读】华为大学的执行校长为陈海燕。2013年全年完成16万人天的面授课程，约有1500多名兼职讲师。在本次汇报会上，任正非重新明确了华大的责任定位、管理架构 ...


## Step 2 — Chunk the corpus and build a TF-IDF index

A single speech can run to thousands of characters — feeding whole speeches to a model is both too long and too diluted. So we **chunk**: cut each document into ~500-character pieces with an 80-character overlap (so no sentence is severed at a boundary), keeping **year and title** on every chunk for later citation.

We *deliberately* use transparent **TF-IDF sparse retrieval** here rather than a black-box embedding API — first understand the mechanics of RAG; Step 8 discusses why production systems switch to vector databases.

> **TF-IDF in one sentence:** the more a word appears in *this chunk* (TF↑) and the rarer it is in *the whole corpus* (IDF↑), the more it characterizes the chunk. The query and every chunk become "word → weight" vectors, ranked by **cosine similarity**.

In [3]:
CHUNK_SIZE = 500      # characters per chunk
OVERLAP    = 80       # overlap between neighboring chunks (don't sever sentences)

index, chunks, docs = rag_ren.build_index(
    corpus_dir=CORPUS_DIR, backend="tfidf",
    chunk_size=CHUNK_SIZE, overlap=OVERLAP, verbose=False,
)
summary = rag_ren.corpus_summary(docs, chunks)
print(f"documents        : {summary['documents']}")
print(f"chunks           : {summary['chunks']}")
print(f"TF-IDF vocabulary: {len(index.idf):,} tokens")
print(f"chunk_size = {CHUNK_SIZE} | overlap = {OVERLAP}")

documents        : 405
chunks           : 4654
TF-IDF vocabulary: 157,065 tokens
chunk_size = 500 | overlap = 80


> **✏️ Your turn:** change `CHUNK_SIZE` above to `200` (too fine) and `1500` (too coarse), re-run these two cells each time, and watch the chunk count change. Think: how would chunks that are too small / too large affect the *quality of the evidence* retrieved later?

## Step 3 — The ten discussion questions

These are **not** factual lookups — they are strategy-judgment questions a business school would debate. Each comes with: the conventional view, the position Ren actually argues in the corpus (ren view), and several **anchors** (fragments of source filenames the retriever is expected to hit) for auditing whether retrieval grabbed the right material.

The question strings are Chinese **because they are the retrieval queries** against a Chinese corpus; the table shows an English gloss beside each.

In [4]:
try:
    from IPython.display import Markdown, display
    rows = ["| # | Theme | Question (retrieval query, zh) | English gloss |",
            "|---|-------|-------------------------------|---------------|"]
    for it in QUESTIONS:
        q = it["q"]; short = q[:40] + ("…" if len(q) > 40 else "")
        q_en = it["q_en"]; short_en = q_en[:80] + ("…" if len(q_en) > 80 else "")
        rows.append(f"| {it['id']} | {it['theme_en']}<br>({it['theme']}) | {short} | {short_en} |")
    display(Markdown("\n".join(rows)))
except Exception:
    for it in QUESTIONS:
        print(f"{it['id']:>2}. [{it['theme_en']} / {it['theme']}]")
        print(f"    zh: {it['q']}")
        print(f"    en: {it['q_en']}")

| # | Theme | Question (retrieval query, zh) | English gloss |
|---|-------|-------------------------------|---------------|
| 1 | US-China tech war / self-reliance<br>(中美科技战 / 自主可控) | 面对美国的技术封锁和实体清单，中国科技公司应该全面'去美化'实现完全自主可控，还… | Facing US technology blockades and the Entity List, should Chinese tech companie… |
| 2 | 5G strategy<br>(5G 战略) | 5G 真的是基础设施级别的革命吗？运营商和企业应该激进部署还是观望等待杀手应用？ | Is 5G really an infrastructure-level revolution? Should carriers and enterprises… |
| 3 | Downturn management / R&D spending<br>(周期管理 / 研发投入) | 经济下行或制裁压力下，科技公司应该裁员降本以保住现金流，还是继续加大研发投入？ | Under a downturn or sanctions pressure, should a tech company cut staff and cost… |
| 4 | Decision philosophy / 'grayscale'<br>(决策哲学 / 灰度) | 管理者面对复杂战略决策时，应该追求数据驱动的最优解，还是接受'方向大致正确即可'… | Facing complex strategic decisions, should managers pursue the data-driven optim… |
| 5 | Investing in basic research<br>(基础研究投资) | 公司应不应该花重金请基础研究科学家（数学家、物理学家），即使他们短期内不产出商业… | Should a company spend heavily on basic-research scientists (mathematicians, phy… |
| 6 | Talent management / veteran employees<br>(人才管理 / 老员工) | 如何处理表现不再适应岗位、跟不上技术演进的老员工？是优化淘汰还是再赋能？ | What to do with veteran employees who no longer fit their roles or keep up with … |
| 7 | Corporate governance / going public<br>(公司治理 / 上市) | 创始人应不应该让公司上市、引入外部资本，并最终交班给职业经理人？ | Should a founder take the company public, bring in outside capital, and eventual… |
| 8 | Organization design / big-company disease<br>(组织设计 / 大企业病) | 公司从几千人增长到几万、几十万人后，如何避免'大企业病'——决策迟缓、内耗、创新… | As a company grows from thousands to hundreds of thousands of people, how do you… |
| 9 | Competitive strategy<br>(竞争战略) | 面对成熟市场的强大对手，企业应该差异化（蓝海、避开正面竞争）还是正面对抗（资源压… | Against strong incumbents in a mature market, should a firm differentiate (blue … |
| 10 | Motivating top talent<br>(顶级人才激励) | 顶级技术人才靠什么留住？是高薪、股权、使命感，还是别的什么？ | What actually retains top technical talent — pay, equity, mission, or something … |

## Step 4 — Audit retrieval quality first (before any generation)

Good engineering habit: **look at retrieval before you talk about generation**. Below we run the anchor diagnostic for every question — if at least one expected anchor shows up in the top-6, retrieval is on track. Note that **no model has been called yet**.

In [5]:
diagnostics = rag_ren.retrieval_diagnostics(index, QUESTIONS, top_k=6)
theme_en = {it["id"]: it["theme_en"] for it in QUESTIONS}
for row in diagnostics:
    status = "PASS" if row["anchor_hit_count"] >= 1 else "FAIL"
    print(f"{status} | Q{row['id']:02d} | top_score={row['top_score']:.3f} | "
          f"anchors={row['anchor_hit_count']}/{row['anchor_count']} | {theme_en[row['id']]}")
all_ok = all(r["anchor_hit_count"] >= 1 and r["top_score"] > 0.01 for r in diagnostics)
print(f"\nall retrieval checks passed: {all_ok}")

PASS | Q01 | top_score=0.151 | anchors=2/4 | US-China tech war / self-reliance
PASS | Q02 | top_score=0.205 | anchors=2/6 | 5G strategy
PASS | Q03 | top_score=0.164 | anchors=1/4 | Downturn management / R&D spending
PASS | Q04 | top_score=0.233 | anchors=2/2 | Decision philosophy / 'grayscale'
PASS | Q05 | top_score=0.288 | anchors=1/3 | Investing in basic research
PASS | Q06 | top_score=0.143 | anchors=1/4 | Talent management / veteran employees
PASS | Q07 | top_score=0.157 | anchors=1/3 | Corporate governance / going public
PASS | Q08 | top_score=0.120 | anchors=2/6 | Organization design / big-company disease
PASS | Q09 | top_score=0.170 | anchors=2/5 | Competitive strategy
PASS | Q10 | top_score=0.167 | anchors=1/3 | Motivating top talent

all retrieval checks passed: True


## Step 5 — Walk through one question: what does the retrieved "evidence" look like?

Change `QUESTION_ID` to switch questions. For now we **only look at the top chunks — no answer is generated**. Judge like a referee: is this material relevant, and is it *enough* to support a well-grounded answer?

In [6]:
QUESTION_ID = 4        # 4 = grayscale / decision philosophy; try 1..10
TOP_K = 6

item = next(it for it in QUESTIONS if it["id"] == QUESTION_ID)
question = item["q"]                     # the Chinese retrieval query
retrieved = index.search(question, top_k=TOP_K)

print(f"Q{QUESTION_ID} [{item['theme_en']} / {item['theme']}]")
print(f"zh: {question}")
print(f"en: {item['q_en']}\n")
print(f"conventional view: {item['conventional_view_en']}")
print(f"Ren's position   : {item['ren_view_en']}\n")
print("top retrieved chunks:\n")
for rank, r in enumerate(retrieved, 1):
    c = r.chunk
    preview = " ".join(c.text.split())[:240]
    print(f"{rank}. score={r.score:.3f}  [{c.year} · {c.title}]")
    print(f"   source: {c.source_file}")
    print(f"   {preview}…\n")

hits = rag_ren.hits_anchor(retrieved, item.get("anchors", []), top_n=TOP_K)
print(f"anchor hits: {len(hits)}/{len(item.get('anchors', []))}  {hits}")

Q4 [Decision philosophy / 'grayscale' / 决策哲学 / 灰度]
zh: 管理者面对复杂战略决策时，应该追求数据驱动的最优解，还是接受'方向大致正确即可'的灰度判断？
en: Facing complex strategic decisions, should managers pursue the data-driven optimum, or accept 'grayscale' judgment — roughly the right direction is enough?

conventional view: Data-driven, find local optima, iterate.
Ren's position   : The grayscale philosophy: roughly-right direction plus a vital organization; compromise is wisdom, not weakness; over-precision breeds rigidity.

top retrieved chunks:

1. score=0.233  [2017 · 在人力资源管理纲要2.0沟通会上的讲话]
   source: 2017/20170807_在人力资源管理纲要2.0沟通会上的讲话.md
   放心。我们只要处理好这个问题，在客户授权下，聚焦在ICT基础设施和智能终端，是可以有作为的。战略不要发散，ICT基础设施和智能终端已经是很大很复杂的业务领域。 未来的价值创造来源“以客户需求和技术创新双轮驱动”，我是同意的。商业模式创新是一个工具，目标还是满足客户需求。首先我们要不断地形成方向大致正确、组织充满活力，就能胜出。如果方向不正确，是产生不出价值来的，组织也难以充满活力。方向正确是领袖要素。领袖要素是方向大致正确的一个保障，组织充满活力要成为方向大致正确的另一个保障…

2. score=0.230  [2017 · 方向要大致正确，组织要充满活力]
   source: 2017/20170602_方向要大致正确，组织要充满活力.md
   ## 方向要大致正确，组织要充满活力 ——任正非在公司战略务虚会上的讲话 2017年6月2日-4日 【导 读】华为蓝军部长潘

> **✏️ Your turn:** write **your own question** in the cell below (Chinese retrieves best against this Chinese corpus). Try a topic the corpus *should* contain (R&D spending, going public, the "Genius Youth" program) and one it *shouldn't* (say, metaverse valuations), and compare the `top_score` of the two.

In [7]:
MY_QUESTION = "华为为什么坚持不上市？"     # <- your question ("Why does Huawei refuse to go public?")
# Chinese works best here: the corpus is Chinese, and TF-IDF needs the words to match.

for rank, r in enumerate(index.search(MY_QUESTION, top_k=5), 1):
    c = r.chunk
    print(f"{rank}. score={r.score:.3f}  [{c.year} · {c.title}]")
    print(f"   {' '.join(c.text.split())[:160]}…")

1. score=0.171  [2013 · 家人永远不接班]
   ## 家人永远不接班 ——任正非在持股员工代表大会的发言摘要 2013年3月30日 【导 读】在会议的各项决议表决后，任正非离题说了三个方面的问题：一，关于公司上市的传闻问题；二，关于接班人传闻问题；三，关于与媒体的关系问题。 一、关于公司上市问题的澄清 任何公司的发展是不是只有上市一条路，允不允许一些企业缓慢地积累增…
2. score=0.166  [2006 · 天道酬勤]
   ## 天道酬勤 【导 读】到2005年，华为虽然在国内市场取得了领先地位，但是与国际巨头相比还很弱小，因此，任正非认为华为还没有成功，还需要艰苦奋斗。华为开始针对“高层要有使命感，中层要有危机感，基层要有饥饿感”的奋斗导向，进行分层驱动。 华为正处在一个关键的发展时期，我们已经连续数年大量招收新员工，壮大队伍。新员工进…
3. score=0.156  [2014 · 英国媒体会谈纪要]
   ，2018年华为的销售收入有可能达到700亿—800亿美金。 10、华尔街日报Sam Schechner:只是再确认一下，股市是贪婪的，意味着华为不会上市？ 任正非：华为公司也会是贪婪的，我们只是尽力抑制。我们在一段时间里不上市，但我们不能保证，我们永远不上市。“永远不上市”这句话，在逻辑上是不通的，因为生命不能永远，…
4. score=0.139  [2014 · 握紧拳头才有力量]
   没有看到，能做太平洋这么粗管道“铁皮”的公司已经没几家了，我们一定是胜利者。所以要坚定一个信心，华为是不是互联网公司并不重要，华为的精神是不是互联网精神也不重要，这种精神能否使我们活下去，才是最重要的。乌龟就是坚定不移往前走，不要纠结、不要攀附，坚信自己的价值观，坚持合理的发展，别隔山羡慕那山的花。 ④有员工在公司心声…
5. score=0.128  [2014 · 英国媒体会谈纪要]
   澄清不会有这么大的作用，还是要用行动来证明。 15、彭博社Sam Chambers: 上市是不是有助于缓解和应对美国的担心？是不是能够帮助华为打入美国市场？ 任正非：是否进入美国市场，不是透明度的问题。即使透明了，也解决不了美国的担忧。现在找不到华为哪点不透明，因为华为每一年的财务报表都有披露，因此透明度和上市本身没有…


## Step 6 — Generate a grounded answer with citations (offline, no LLM)

This step is **extractive**: pick the most relevant sentences from the retrieved chunks and attach a `[year · title]` citation to each. Less fluent than a large model — but **every sentence traces back to evidence**. That is RAG's core promise: **citations are part of the answer's contract, not decoration**.

First, look at how the **prompt** fed to a model would be assembled: instructions + retrieved context + user question. (The prompt template is intentionally Chinese — it instructs the model to answer from a Chinese corpus.)

In [8]:
prompt = rag_ren.build_prompt(question, retrieved)
print(prompt[:700], "\n… (truncated) …\n")

print("="*60)
result = rag_ren.compare(question, index, top_k=TOP_K, llm_backend="extractive")
print("question:", result["question"])
print("\nextractive grounded answer (every line carries a citation):\n")
print(result["rag"])

你是一个严谨的中文问答助手，专门基于任正非历年讲话回答问题。
请只根据下面【检索上下文】回答【用户问题】。
要求：
1. 引用具体年份和讲话标题，例如 [2019 · 接受金融时报采访]。
2. 如果上下文中没有充分答案，请明确说明 “检索语料中没有找到直接证据”，不要编造。
3. 用简体中文回答。

【检索上下文】
[2017 · 在人力资源管理纲要2.0沟通会上的讲话] (chunk 4083, score=0.233)
放心。我们只要处理好这个问题，在客户授权下，聚焦在ICT基础设施和智能终端，是可以有作为的。战略不要发散，ICT基础设施和智能终端已经是很大很复杂的业务领域。

未来的价值创造来源“以客户需求和技术创新双轮驱动”，我是同意的。商业模式创新是一个工具，目标还是满足客户需求。首先我们要不断地形成方向大致正确、组织充满活力，就能胜出。如果方向不正确，是产生不出价值来的，组织也难以充满活力。方向正确是领袖要素。领袖要素是方向大致正确的一个保障，组织充满活力要成为方向大致正确的另一个保障。组织充满活力既要能够使得大致正确的方向得以贯彻执行，也要善于自我批判，使得一旦方向脱离大致正确后,能够及时纠偏。在知识爆炸、行业快速变化的今天，充满活力的组织要让领袖听得见来自各个层级的声音，吸收全组织的精华，以保证持续维持大致正确的方向。

[2017 · 方向要大致正确，组织要充满活力] (chunk 3999, score=0.230)
## 方向要大致正确，组织要充满活力

——任正非在公司战略务虚会上的讲话
2017年6月2日-4日



【导  读】华为蓝军部长潘少钦透 
… (truncated) …

question: 管理者面对复杂战略决策时，应该追求数据驱动的最优解，还是接受'方向大致正确即可'的灰度判断？

extractive grounded answer (every line carries a citation):

根据检索，最相关的证据片段是 (top evidence sentences from retrieval):
- 对此，任总做了两点澄清和解释，首先，这里的方向是指产业方向和技术方向，我们不可能完全看的准，做到大致准确就很了不起；其次，在方向大致准确的情况，组织充满活力非常重要，是确保战略执行、走向成功的关键。 [201

## Step 7 — ✏️ Build a minimal retriever yourself

TF-IDF sounds fancy, but the core idea is humble: **split the question and each chunk into tokens, count how many tokens they share, rank by the count.** Implement this "baby retriever" `overlap_search` below, then compare its top-1 against `index` (the real TF-IDF).

You will need `rag_ren.tokenize(text)` — it tokenizes mixed Chinese/English (Chinese as characters + bigrams). **Write it yourself first; the reference solution is in the Appendix.**

In [9]:
def overlap_search(query, chunks, top_k=6):
    """Score each chunk by "number of tokens shared with the query"; return top_k (score, chunk).
    Step hints:
      1. q_tokens = set(rag_ren.tokenize(query))
      2. for each chunk: c_tokens = set(rag_ren.tokenize(chunk.text))
      3. score = size of the intersection  ->  len(q_tokens & c_tokens)
      4. collect (score, chunk), sort by score descending
      5. return the first top_k
    (Reference solution in the Appendix.)
    """
    # your code here
    raise NotImplementedError("Implement overlap_search — reference solution in the Appendix.")


# -- once implemented, run this to compare against the real TF-IDF --
try:
    mine = overlap_search(question, chunks, top_k=3)
    print("your overlap_search top-1:", rag_ren.format_citation(mine[0][1]))
    print("TF-IDF          top-1    :", rag_ren.format_citation(index.search(question, top_k=1)[0].chunk))
    print("\nDiscuss: same or different? If different, think about how IDF down-weights",
          "\nhigh-frequency words like 我们/公司/发展 ('we'/'company'/'development').")
except NotImplementedError as e:
    print("not implemented yet:", e)

not implemented yet: Implement overlap_search — reference solution in the Appendix.


## Step 8 (optional) — Semantic embedding retrieval: when keywords don't match

TF-IDF's weak spot: it needs a **literal word match**. Ask about 灰度管理 ("grayscale management") when the corpus says 在黑白之间保留妥协与宽容 ("keep compromise and tolerance between black and white") — zero shared words, so TF-IDF comes up empty.

An **embedding** maps any text to a dense vector $E:\text{text}\to\mathbb{R}^d$ so that **texts with similar meaning get nearby vectors** — even with no word overlap.

> ⚠️ **This section is optional and needs to download a ~120 MB model** (first run only). Whether the download works in mainland China depends on your network; if it doesn't, **nothing earlier is affected**. Install: `pip install -i https://pypi.tuna.tsinghua.edu.cn/simple sentence-transformers`

In [10]:
import time
EMBED_OK = False
try:
    embed_index = rag_ren.LocalEmbeddingIndex(
        chunks=chunks,
        embedding_model=rag_ren.DEFAULT_LOCAL_EMBED_MODEL,
        cache_path=BUILD_DIR / "local_embedding_cache.json",
    )
    t0 = time.time()
    embed_index.build(verbose=True)          # first run: download + encode; cached afterwards
    print(f"\n[embed] ready in {time.time()-t0:.1f}s; vector dimension = {len(embed_index.chunk_vectors[0])}")
    EMBED_OK = True
except Exception as exc:
    print(f"[embed] skipped ({type(exc).__name__}: {exc})")
    print('To enable: pip install -i https://pypi.tuna.tsinghua.edu.cn/simple sentence-transformers, then re-run this cell.')

[embed] skipped (RuntimeError: sentence-transformers is not installed. Install with: pip install -e ".[embed]")
To enable: pip install -i https://pypi.tuna.tsinghua.edu.cn/simple sentence-transformers, then re-run this cell.


In [11]:
# For the same question, compare TF-IDF vs embedding top-1; runs only if embeddings are ready.
if EMBED_OK:
    tf1 = index.search(question, top_k=1)[0]
    em1 = embed_index.search(question, top_k=1)[0]
    print(f"question: {question}\n")
    print(f"TF-IDF    top-1: score={tf1.score:.3f}  {rag_ren.format_citation(tf1.chunk)}")
    print(f"Embedding top-1: score={em1.score:.3f}  {rag_ren.format_citation(em1.chunk)}")
    print("\nNote: swapping the retrieval backend changed NOTHING else —",
          "build_prompt, the extractive answer, and the citation format are untouched.",
          "Only the 'retrieve' layer was replaced.")
else:
    print("Skipped — install and build the embedding index in the previous cell first.")

Skipped — install and build the embedding index in the previous cell first.


## Step 9 (optional) — naive LLM vs RAG: show students the difference

**Off by default.** To demo the difference between a bare model (no retrieval) and RAG in class, set `ENABLE_LLM = True`. This calls the local `claude -p`, so you must have run `claude` in a terminal and completed `/login` first.

In [12]:
ENABLE_LLM = False        # set True only after running `claude` + /login in a terminal
LLM_MODEL = "sonnet"

if ENABLE_LLM:
    ok, msg = rag_ren.claude_code_auth_check(timeout=30)
    print(msg)
    if not ok:
        raise RuntimeError("Claude Code not ready: run `claude` in a terminal and /login.")
    res = rag_ren.compare(question, index, top_k=TOP_K,
                          llm_backend="claude-code", llm_model=LLM_MODEL)
    print("\n----- Naive (no retrieval) -----\n", res["naive"])
    print("\n----- RAG (with retrieved context) -----\n", res["rag"])
else:
    print("Skipped. Set ENABLE_LLM = True to run the optional Claude Code comparison.")

Skipped. Set ENABLE_LLM = True to run the optional Claude Code comparison.


## Class discussion / wrap-up

Suggested order:
1. Have students answer from **intuition** first (the conventional view).
2. Show the **top retrieved chunks**; let students judge whether the evidence is relevant and sufficient.
3. Show the **extractive grounded answer**; stress the **citation contract**: every line traces to a year and a speech.
4. *(Optional)* run Claude Code and compare naive vs RAG answers.
5. Push further: if we swap the retriever for embeddings, **which layer of the RAG architecture changes, and which layers don't?**

**Key takeaway:** RAG = *retrieval* + *generation*. It doesn't touch the model's weights — it **changes the available evidence** and **constrains the answer to an auditable corpus**. For more depth, see Lecture 8 T2 (chunking & embeddings) and T3 (vector retrieval / GraphRAG).

### Make it your own
- Swap in your own corpus (a folder of `.md` files; point `REN_CORPUS_DIR` at it) and re-run the whole pipeline.
- Add a strategy question you care about to `questions.py` — with its `q_en` gloss and its anchors.
- Upgrade `overlap_search`: down-weight high-frequency tokens (the IDF idea) and see whether the ranking moves closer to TF-IDF.

---
## Appendix — Reference solution

Write your own Step 7 first, then look here.

In [13]:
# --- Reference solution: overlap_search ---
def overlap_search(query, chunks, top_k=6):
    q_tokens = set(rag_ren.tokenize(query))
    scored = []
    for c in chunks:
        c_tokens = set(rag_ren.tokenize(c.text))
        score = len(q_tokens & c_tokens)          # shared token count
        scored.append((score, c))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

mine = overlap_search(question, chunks, top_k=3)
print("overlap_search top-3:")
for score, c in mine:
    print(f"  overlap={score:>3}  {rag_ren.format_citation(c)}")
print("\nTF-IDF top-1:", rag_ren.format_citation(index.search(question, top_k=1)[0].chunk))
print("\nWhy they can differ: overlap only counts shared tokens, so high-frequency words"
      "\nlike 我们/公司/发展 ('we'/'company'/'development') flood the score; TF-IDF's IDF"
      "\nterm down-weights them, favoring discriminative words like 灰度/妥协"
      "\n('grayscale'/'compromise').")

overlap_search top-3:
  overlap= 34  [2013 · 用乌龟精神，追上龙飞船]
  overlap= 32  [2006 · 在华为收购港湾时的谈话纪要]
  overlap= 32  [2014 · 在“班长的战争”对华为的启示和挑战汇报会上的讲话]

TF-IDF top-1: [2017 · 在人力资源管理纲要2.0沟通会上的讲话]

Why they can differ: overlap only counts shared tokens, so high-frequency words
like 我们/公司/发展 ('we'/'company'/'development') flood the score; TF-IDF's IDF
term down-weights them, favoring discriminative words like 灰度/妥协
('grayscale'/'compromise').
